In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 10
fig_height = 5
fig_format = 'retina'
fig_dpi = 96
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL3NpZHYvZ2l0aHViL3NsaWRlcy91dmEtMjAyNg=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/Users/sidv/.local/share/uv/python/cpython-3.13.3-macos-aarch64-none/lib/python3.13/importlib/_bootstrap.py": 1767399579.7657862, "/Users/sidv/.local/share/uv/python/cpython-3.13.3-macos-aarch64-none/lib/python3.13/importlib/_bootstrap_external.py": 1767399579.7660959, "/Users/sidv/.local/share/uv/python/cpython-3.13.3-macos-aarch64-none/lib/python3.13/zipimport.py": 1767399579.8654232, "/Users/sidv/.local/share/uv/python/cpython-3.13.3-macos-aarch64-none/lib/python3.13/codecs.py": 1767399579.6528504, "/Users/sidv/.local/share/uv/python/cpython-3.13.3-macos-aarch64-none/lib/python3.13/encodings/aliases.py": 1767399579.6670706, "/Users/sidv/.local/share/uv/python/cpython-3.13.3-macos-aarch64-none/lib/python3.13/encodings/__init__.py": 1767399579.666475, "/Users/sidv/.local/share/uv/python/cpython-3.13.3-macos-aarch64-none/lib/python3.13/encodings/utf_8.py": 1767399579.6797054, "/Users/sidv/.local/share/uv/python/cpython-3.13.3-macos-aarch64-none/lib/python3.13/abc.py": 1767399579.6457

In [2]:
import plotly.graph_objects as go
import numpy as np

# your matrix
pts = np.array([
  [33.395451954774344, -116.56439161038234],
[48.108025710076475, -122.92849274278078],
[34.02461082128876, -112.71933666708098],
[35.61729882163741, -114.72833547890279],
[45.85967498821038, -109.49033666914245],
[38.37289482050906, -90.24731696226311],
[42.228117661145966, -87.1895050727579],
[41.375254336650364, -82.52162610658272],
[40.27227181552411, -81.31719495644117],
[41.080749321883864, -73.2409874547851],
[29.799471507417657, -87.93195684760049],
[29.804262181778334, -96.26280221549924],
[32.255403145659265, -97.63920129263443],
[34.86077879074841, -97.48276955656243],
[37.55028764727703, -98.50201856867726],
[38.669659368654706, -78.72114542890988],
[42.198688371914685, -72.50471116858739],
[33.37589451879909, -82.33537390544765],
[28.66713158331832, -82.40762693948109],
[35.50496894714642, -105.32059009034614],
[39.311036495683894, -103.23348820810236],
[40.37184108073199, -113.49645343304725],
[38.151089336480155, -122.23163700966431],
[41.379564699954805, -106.00341934279291],
[40.96859401421174, -94.56363956247374],
[45.33417319402169, -94.1289712611423],
[36.210756845201466, -86.66439950758569],
[31.910335493320844, -78.73442885210514],
[32.16531293340556, -105.36009678147337],
[35.05389959257307, -93.93624630674735],
])

cities = {
    "San Diego": (32.7157, -117.1611),
    "Seattle": (47.6062, -122.3321),
    "Phoenix": (33.4484, -112.0740),
    "Las Vegas": (36.1699, -115.1398),
    "Bozeman": (45.6760, -111.0429),
    "St Louis": (38.6270, -90.1994),
    "Chicago": (41.8781, -87.6298),
    "Cleveland": (41.4993, -81.6944),
    "Pittsburgh": (40.4406, -79.9959),
    "New York": (40.7128, -74.0060),
    "New Orleans": (29.9511, -90.0715),
    "Houston": (29.7604, -95.3698),
    "Dallas": (32.7767, -96.7970),
    "Oklahoma City": (35.4676, -97.5164),
    "Wichita": (37.6872, -97.3301),
    "Charlottesville": (38.0293, -78.4767),
    "Boston": (42.3601, -71.0589),
    "Atlanta": (33.7490, -84.3880),
    "Orlando": (28.5383, -81.3792),

    # New additions
    "Santa Fe": (35.6870, -105.9378),
    "Denver": (39.7392, -104.9903),
    "Salt Lake City": (40.7608, -111.8910),
    "San Francisco": (37.7749, -122.4194),
    "Cheyenne": (41.1400, -104.8202),
    "Omaha": (41.2565, -95.9345),
    "Minneapolis": (44.9778, -93.2650),
    "Nashville": (36.1627, -86.7816),
    "Charleston": (32.7765, -79.9311),
    "El Paso": (31.7619, -106.4850),
    "Little Rock": (34.7465, -92.2896)
}

lats = [cities[c][0] for c in cities]
lons = [cities[c][1] for c in cities]
labels = list(cities.keys())

fig = go.Figure()

a = 240
fig.update_layout(
    geo=dict(
        scope="usa",
        projection=go.layout.geo.Projection(type="albers usa"),
        showland=True,
        landcolor=f"rgba({a}, {a}, {a}, 1.0)",
        showsubunits=True,
        subunitcolor="black",   # state border color
        subunitwidth=1
    )
)

fig.add_trace(go.Scattergeo(
    lat=pts[:,0],
    lon=pts[:,1],
    mode="markers",
    marker=dict(
        symbol="circle",
        size=10,
        color="black",
        line=dict(width=1, color="black")
    ),
    hoverinfo="skip"        # disables hover
))

fig.add_trace(go.Scattergeo(
    lon=lons,
    lat=lats,
    text=labels,
    mode="markers+text",
    textposition="top center",
    marker=dict(
        size=10,
        symbol="x",
        color="red",
        line=dict(width=0.01, color="red")
    ),
    hoverinfo="skip"        # disables hover
))

fig.update_geos(
    lataxis=dict(range=[24, 50]),
    lonaxis=dict(range=[-125, -66])
)
# adjust figure size



fig.show()

In [3]:
ellipses = [
    np.array([[2.468017077579583, -0.39142097029727185], [-0.39142097029727185, 10.289982779036782]]),
    np.array([[2.250906098723061, 0.11238004050197975], [0.11238004050197975, 7.588772628436054]]),
    np.array([[1.1927962770755824, -0.03172295061944705], [-0.03172295061944705, 8.259885002626435]]),
    np.array([[1.8406536008076417, 0.43800337694479713], [0.43800337694479713, 10.452625908907532]]),
    np.array([[1.5944867619391174, 0.33968161933149954], [0.33968161933149954, 4.767834664449261]]),
    np.array([[1.21881992369177, 0.2013772476496925], [0.2013772476496925, 3.2307681191014104]]),
    np.array([[3.891984094523177, 0.07374895930561526], [0.07374895930561526, 9.320467587921016]]),
    np.array([[1.4546459336890705, -0.24982534642073528], [-0.24982534642073528, 3.8920963311497423]]),
    np.array([[3.8432371387420385, -0.1430584400033656], [-0.1430584400033656, 10.309892186856777]]),
    np.array([[1.5088822387315137, -0.5023502631328222], [-0.5023502631328222, 4.748962962327717]]),
    np.array([[4.263602342057142, -1.6999153867114396], [-1.6999153867114396, 9.034773668248434]]),
    np.array([[1.046934186691528, 0.19225180857869922], [0.19225180857869922, 4.843336671539482]]),
    np.array([[0.783520265070989, 0.413563369927824], [0.413563369927824, 3.030009405731471]]),
    np.array([[1.6924083470669555, -0.11296900271252386], [-0.11296900271252386, 6.986292520542568]]),
    np.array([[2.753018506557935, -0.49349319570788175], [-0.49349319570788175, 7.639370563223294]]),
    np.array([[1.279775680238266, 0.3666922113139228], [0.3666922113139228, 4.80871214220213]]),
    np.array([[1.708449169704774, -0.4454344166130284], [-0.4454344166130284, 4.783876481597376]]),
    np.array([[2.1113931780514754, -0.3333267899683792], [-0.3333267899683792, 9.511067085731291]]),
    np.array([[1.0955276360627721, 0.5103416488925722], [0.5103416488925722, 4.544539746936275]]),
    np.array([[1.1431929896672604, 0.6707389617601615], [0.6707389617601615, 5.094603284354269]]),
    np.array([[2.005688002020819, -0.48802726217223813], [-0.48802726217223813, 5.0339543953624]]),
    np.array([[1.939711341768792, -0.5436713780240532], [-0.5436713780240532, 5.788115319419898]]),
    np.array([[3.9778613616532974, 0.21292670375766287], [0.21292670375766287, 10.31393281462724]]),
    np.array([[1.535735186385431, 0.5095400023738197], [0.5095400023738197, 5.414367425750979]]),
    np.array([[1.1119111955083785, -0.5197743259158305], [-0.5197743259158305, 1.9814023823549625]]),
    np.array([[2.5329409225953916, -0.18359025032545156], [-0.18359025032545156, 10.52380605919815]]),
    np.array([[1.2328875577305305, 0.2437246475835416], [0.2437246475835416, 5.745240329350703]]),
    np.array([[2.004979908731868, -0.060380495187101466], [-0.060380495187101466, 7.286021040411914]]),
    np.array([[2.2854068119779853, -0.5217193598799783], [-0.5217193598799783, 5.163064894484483]]),
    np.array([[2.265361526692174, -0.3768095912405867], [-0.3768095912405867, 7.1336234243512635]]),
]

In [4]:
import numpy as np
import plotly.graph_objects as go

# pts: (N,2) array of [lat, lon]
# ellipses: list of N (2,2) SPD "shape" matrices

def ellipse_latlon_points(center_latlon, S, n=80, level=1.0):
    """
    Unit ellipsoid/ellipse defined by (x-c)^T S (x-c) = level^2.
    Here x is [lat, lon] in degrees. Returns arrays (lats, lons).

    If you intended S to be a covariance matrix instead, use S_inv = np.linalg.inv(S)
    (see note at bottom).
    """
    c = np.asarray(center_latlon, dtype=float)          # [lat, lon]
    S = np.asarray(S, dtype=float)
    S = np.linalg.inv(S)

    # For (x-c)^T S (x-c) = level^2, we can write:
    # x = c + (level) * inv(sqrtm(S)) @ u,  where ||u||=1
    # Compute inv(sqrtm(S)) via eigen-decomposition (S is 2x2 SPD).
    w, V = np.linalg.eigh(S)
    inv_sqrt = V @ np.diag(1.0 / np.sqrt(w)) @ V.T      # 2x2

    t = np.linspace(0, 2*np.pi, n, endpoint=True)
    unit_circle = np.vstack([np.cos(t), np.sin(t)])     # 2 x n

    pts_local = c[:, None] + (level * inv_sqrt @ unit_circle)
    lats = pts_local[0, :]
    lons = pts_local[1, :]
    return lats, lons

fig = go.Figure()

a = 240
fig.update_layout(
    geo=dict(
        scope="usa",
        projection=go.layout.geo.Projection(type="albers usa"),
        showland=True,
        landcolor=f"rgba({a}, {a}, {a}, 1.0)",
        showsubunits=True,
        subunitcolor="black",
        subunitwidth=1
    )
)

# original points
fig.add_trace(go.Scattergeo(
    lat=pts[:, 0],
    lon=pts[:, 1],
    mode="markers",
    marker=dict(symbol="circle", size=10, color="black", line=dict(width=1, color="black")),
    hoverinfo="skip"
))

# ellipses as line loops
for i, (c, S) in enumerate(zip(pts, ellipses)):
    elat, elon = ellipse_latlon_points(c, S, n=90, level=1.0)
    fig.add_trace(go.Scattergeo(
        lat=elat,
        lon=elon,
        mode="lines",
        line=dict(width=2, color="rgba(0,0,0,0.45)"),
        hoverinfo="skip",
        showlegend=False,
        name=f"ell_{i}"
    ))

# (optional) city markers from your existing code
fig.add_trace(go.Scattergeo(
    lon=lons,
    lat=lats,
    text=labels,
    mode="markers+text",
    textposition="top center",
    marker=dict(size=10, symbol="x", color="red", line=dict(width=0.01, color="red")),
    hoverinfo="skip"
))

fig.update_geos(
    lataxis=dict(range=[24, 50]),
    lonaxis=dict(range=[-125, -66])
)

fig.show()

# NOTE:
# If your "shape matrix" is actually a covariance matrix Σ and you want the usual covariance ellipse
# (x-c)^T Σ^{-1} (x-c) = level^2, then call:
#   elat, elon = ellipse_latlon_points(c, np.linalg.inv(Sigma), level=1.0)
# or change ellipse_latlon_points to use sqrtm(Σ) instead of inv(sqrtm(S)).